# ♻️ Kabadiwala — 16-Class Zero-Bias Fast Scrap Vision Training Pipeline
**TrashBox-testandvalid + TrashNet | Fast-Track (200/class) | Early Stopping | MobileNetV3 & DINOv2**

Optimized for fast execution on Google Colab (under 6–8 minutes). Skips oversized multi-gigabyte repos and uses the curated `TrashBox-testandvalid` (e-waste chips, cables, laptops, phones, appliances) + `TrashNet` + field images, balanced to 200 samples per class.

### Table of Contents
- **01** — Environment Setup & Drive Mount
- **02** — Fast Dataset Ingestion (TrashBox-testandvalid + TrashNet in < 90 seconds)
- **03** — 16-Class Taxonomy Mapping & Audit
- **04** — Zero-Bias Balancing (200/class) & Stratified Split (70/15/15)
- **05** — Class-Weighted Loss & PyTorch DataLoaders
- **06** — Train Fast Model: MobileNetV3-Large (Early Stopping, ~3 mins)
- **07** — Train Flagship Model: DINOv2 ViT-S/14 (Early Stopping, ~2.5 mins)
- **08** — Per-Class Evaluation & Confusion Matrix
- **09** — ONNX Export & Dynamic Class Index Sync


In [ ]:
# ============================================================
# SECTION 01 — Environment Setup & Mount
# ============================================================
import os, sys, json, random, shutil, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Kabadiwala_ML'
    IN_COLAB = True
    print('Google Drive mounted.')
except Exception:
    BASE = './Kabadiwala_ML'
    IN_COLAB = False
    print('Running locally without Drive mount.')

DATA_DIR  = os.path.join(BASE, 'data')
RAW_DIR   = os.path.join(DATA_DIR, 'raw')
SPLIT_DIR = os.path.join(BASE, 'splits')
MODEL_DIR = os.path.join(BASE, 'models')
LOG_DIR   = os.path.join(BASE, 'logs')

for d in [DATA_DIR, RAW_DIR, SPLIT_DIR, MODEL_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Setup complete. Working directories initialized.')


In [ ]:
# Install required dependencies
!pip install -q timm ImageHash scikit-learn pandas numpy matplotlib seaborn pillow tqdm onnx onnxruntime

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')


## SECTION 02 — Fast Dataset Ingestion (< 90 seconds)
Downloads:
1. **TrashBox-testandvalid** (Lightweight shallow clone with e-waste chips, wires, laptops, smartphones, small appliances, cardboard, glass, metal, paper, plastic)
2. **TrashNet** (Fast zip download with clean recyclables)
*(Skips the oversized multi-gigabyte main repo to prevent Colab disconnection)*

In [ ]:
# ============================================================
# SECTION 02 — Fast Dataset Download (< 90 seconds)
# ============================================================
import urllib.request, zipfile

TRASHBOX_VALID_DIR = os.path.join(RAW_DIR, 'trashbox_testandvalid')
TRASHNET_DIR       = os.path.join(RAW_DIR, 'trashnet')

# 1. TrashBox-testandvalid (Lightweight, ~1 minute shallow clone)
if not os.path.exists(TRASHBOX_VALID_DIR) or len(os.listdir(TRASHBOX_VALID_DIR)) == 0:
    print('Cloning TrashBox-testandvalid (e-waste chips, cables, laptops, appliances)...')
    !git clone --quiet --depth 1 https://github.com/nikhilvenkatkumsetty/TrashBox-testandvalid.git {TRASHBOX_VALID_DIR}
    print('✅ TrashBox-testandvalid ready.')
else:
    print('✅ TrashBox-testandvalid already downloaded.')

# 2. TrashNet (Direct zip download in ~10 seconds)
trashnet_sub = os.path.join(TRASHNET_DIR, 'dataset-resized')
if not os.path.exists(trashnet_sub):
    print('Downloading TrashNet zip (~10 seconds)...')
    os.makedirs(TRASHNET_DIR, exist_ok=True)
    zip_path = os.path.join(TRASHNET_DIR, 'dataset-resized.zip')
    url = 'https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip'
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(TRASHNET_DIR)
    print('✅ TrashNet extracted.')
else:
    print('✅ TrashNet already downloaded.')


## SECTION 03 — 16-Class Taxonomy Mapping & Scanning
Maps all subfolders to the 16 canonical categories covering E-Waste, Metals, Paper/Cardboard, Plastics, Glass, and Wood.

In [ ]:
# ============================================================
# SECTION 03 — 16-Class Taxonomy Mapping & Scanning
# ============================================================
TARGET_CLASSES = [
    'aluminium', 'appliances', 'batteries', 'cables_wires',
    'cardboard', 'copper', 'glass_mirror', 'iron_steel',
    'laptops_computers', 'mixed_plastic', 'mobile_tablets', 'newspaper_paper',
    'pcb_chips', 'pet_plastic', 'tv_monitors_displays', 'wood'
]
CLASS_TO_IDX = {c: i for i, c in enumerate(TARGET_CLASSES)}
IDX_TO_CLASS = {i: c for i, c in enumerate(TARGET_CLASSES)}

CLASS_MAP = {
    'cardboard': 'cardboard',
    'glass': 'glass_mirror',
    'metal': 'iron_steel',
    'paper': 'newspaper_paper',
    'plastic': 'mixed_plastic',
    'electronic chips': 'pcb_chips',
    'electronic_chips': 'pcb_chips',
    'chips': 'pcb_chips',
    'pcb': 'pcb_chips',
    'electrical cables': 'cables_wires',
    'electric wires, cords and cables': 'cables_wires',
    'cables': 'cables_wires',
    'cable': 'cables_wires',
    'wires': 'cables_wires',
    'wire': 'cables_wires',
    'small appliances': 'appliances',
    'appliances': 'appliances',
    'applicances': 'appliances',
    'smartphones': 'mobile_tablets',
    'mobile': 'mobile_tablets',
    'laptops': 'laptops_computers',
    'laptop': 'laptops_computers',
    'tv': 'tv_monitors_displays',
    'monitor': 'tv_monitors_displays',
    'displays': 'tv_monitors_displays',
    'battery': 'batteries',
    'batteries': 'batteries',
    'copper': 'copper',
    'cu': 'copper',
    'aluminium': 'aluminium',
    'can': 'aluminium',
    'iron_steel': 'iron_steel',
    'iron': 'iron_steel',
    'steel': 'iron_steel',
    'newspaper': 'newspaper_paper',
    'pet_plastic': 'pet_plastic',
    'bottle': 'pet_plastic',
    'mirror': 'glass_mirror',
    'wood': 'wood'
}

def scan_image_folder(root_dir, source_name):
    records = []
    root = Path(root_dir)
    if not root.exists(): return records
    for class_dir in root.rglob('*'):
        if not class_dir.is_dir(): continue
        raw_class = class_dir.name.lower().strip()
        mapped = CLASS_MAP.get(raw_class)
        if mapped is None:
            for k, v in CLASS_MAP.items():
                if k in raw_class:
                    mapped = v; break
        if mapped is None or mapped not in TARGET_CLASSES: continue
        imgs = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg'))
        for img_path in imgs:
            records.append({'filepath': str(img_path), 'source': source_name, 'raw_class': raw_class, 'mapped_class': mapped})
    return records

all_recs = []
all_recs.extend(scan_image_folder(TRASHBOX_VALID_DIR, 'TrashBox_TestAndValid'))
all_recs.extend(scan_image_folder(os.path.join(TRASHNET_DIR, 'dataset-resized'), 'TrashNet'))

df_raw = pd.DataFrame(all_recs)
print(f'Total raw images found: {len(df_raw)}')
if len(df_raw) > 0:
    print(df_raw['mapped_class'].value_counts())


## SECTION 04 — Fast Zero-Bias Balancing (200/class) & Stratified Split
Balances all classes to exactly 200 images each (3,200 total images). Trains in ~5–7 minutes on GPU with zero majority-class bias.

In [ ]:
# ============================================================
# SECTION 04 — Fast Balancing (200/class) & Stratified Split
# ============================================================
from sklearn.model_selection import train_test_split
TARGET_PER_CLASS = 200  # Fast-track: 3,200 total images, fast training
balanced_dfs = []

for cls in TARGET_CLASSES:
    cls_sub = df_raw[df_raw['mapped_class'] == cls]
    count = len(cls_sub)
    if count >= TARGET_PER_CLASS:
        sampled = cls_sub.sample(n=TARGET_PER_CLASS, random_state=SEED)
    elif count > 0:
        sampled = cls_sub.sample(n=TARGET_PER_CLASS, replace=True, random_state=SEED)
    else:
        sampled = cls_sub
    balanced_dfs.append(sampled)

df_balanced = pd.concat(balanced_dfs, ignore_index=True)
print(f'Balanced dataset size: {len(df_balanced)} images across 16 classes.')

# Stratified Split: 70% Train, 15% Val, 15% Test
train_val, test_df = train_test_split(df_balanced, test_size=0.15, random_state=SEED, stratify=df_balanced['mapped_class'])
train_df, val_df = train_test_split(train_val, test_size=0.1765, random_state=SEED, stratify=train_val['mapped_class'])

train_df.to_csv(os.path.join(SPLIT_DIR, 'train.csv'), index=False)
val_df.to_csv(os.path.join(SPLIT_DIR, 'val.csv'), index=False)
test_df.to_csv(os.path.join(SPLIT_DIR, 'test.csv'), index=False)
print(f'Splits: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}')


## SECTION 05 — Early Stopping Callback & DataLoaders
Implements an early stopping callback monitoring validation Macro F1 score to terminate training dynamically as soon as the model peaks.

In [ ]:
# ============================================================
# SECTION 05 — DataLoaders & EarlyStopping Helper
# ============================================================
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.metrics import f1_score, classification_report

TRAIN_TRANSFORMS = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.RandomRotation(15),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

EVAL_TRANSFORMS = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ScrapDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['filepath']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), color=(128, 128, 128))
        if self.transform: img = self.transform(img)
        label = CLASS_TO_IDX.get(row['mapped_class'], 0)
        return img, label

train_loader = DataLoader(ScrapDataset(train_df, TRAIN_TRANSFORMS), batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(ScrapDataset(val_df,   EVAL_TRANSFORMS), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(ScrapDataset(test_df,  EVAL_TRANSFORMS), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Class weights for CrossEntropy
counts = train_df['mapped_class'].value_counts()
weights = torch.tensor([len(train_df) / (len(TARGET_CLASSES) * max(1, counts.get(c, 1))) for c in TARGET_CLASSES], dtype=torch.float32).to(device)
weights = weights / weights.mean()

class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.002, mode='max'):
        self.patience = patience; self.min_delta = min_delta; self.mode = mode
        self.best_score = -float('inf') if mode == 'max' else float('inf')
        self.counter = 0; self.early_stop = False; self.best_epoch = 0
    def step(self, current_score, epoch):
        improved = (current_score - self.best_score) > self.min_delta if self.mode == 'max' else (self.best_score - current_score) > self.min_delta
        if improved:
            self.best_score = current_score; self.counter = 0; self.best_epoch = epoch; return True
        else:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
            return False

print('DataLoaders and EarlyStopping ready.')


## SECTION 06 — Train Fast Edge Model: MobileNetV3-Large (~3 mins)
Lightweight, low-latency model optimized for edge/mobile inference (~17MB).

In [ ]:
# ============================================================
# SECTION 06 — Train MobileNetV3-Large
# ============================================================
import torchvision.models as models
import torch.optim as optim

mobilenet = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
mobilenet.classifier[-1] = nn.Linear(mobilenet.classifier[-1].in_features, len(TARGET_CLASSES))
mobilenet = mobilenet.to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.AdamW(mobilenet.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=12)

mb_early = EarlyStopping(patience=3, min_delta=0.002, mode='max')
mb_best_path = os.path.join(MODEL_DIR, 'mobilenetv3_best.pt')

print('Starting MobileNetV3 training with Early Stopping (max 12 epochs)...')
for epoch in range(1, 13):
    mobilenet.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = mobilenet(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    scheduler.step()

    # Validation
    mobilenet.eval()
    v_preds, v_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            preds = mobilenet(imgs).argmax(1).cpu()
            v_preds.extend(preds.tolist()); v_labels.extend(labels.tolist())
    val_acc = sum(p == l for p, l in zip(v_preds, v_labels)) / len(v_labels)
    val_f1 = f1_score(v_labels, v_preds, average='macro', zero_division=0)

    if mb_early.step(val_f1, epoch):
        torch.save(mobilenet.state_dict(), mb_best_path)
        status = '⭐ [BEST SAVED]'
    else:
        status = f'[Patience: {mb_early.counter}/{mb_early.patience}]'
    print(f'[MobileNetV3] Epoch {epoch:02d}/12 | Loss: {total_loss/total:.4f} | ValAcc: {val_acc:.3f} | ValMacroF1: {val_f1:.4f} {status}')
    if mb_early.early_stop:
        print(f'🛑 Early stopping triggered at epoch {epoch}. Best Val F1: {mb_early.best_score:.4f}')
        break


## SECTION 07 — Train Flagship: DINOv2 ViT-S/14 (~2.5 mins)
Self-supervised Vision Transformer with frozen backbone + multi-layer classification head.

In [ ]:
# ============================================================
# SECTION 07 — Train DINOv2 ViT-S/14
# ============================================================
dinov2_backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', trust_repo=True)
for p in dinov2_backbone.parameters():
    p.requires_grad = False

class DINOv2ScrapClassifier(nn.Module):
    def __init__(self, backbone, num_classes=len(TARGET_CLASSES)):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(384, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        with torch.no_grad():
            feats = self.backbone(x)
        return self.classifier(feats)

dino_model = DINOv2ScrapClassifier(dinov2_backbone, len(TARGET_CLASSES)).to(device)
optimizer = optim.AdamW(dino_model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=12)
dino_early = EarlyStopping(patience=3, min_delta=0.002, mode='max')
dino_best_path = os.path.join(MODEL_DIR, 'dinov2_vits14_best.pt')

print('Starting DINOv2 training with Early Stopping (max 12 epochs)...')
for epoch in range(1, 13):
    dino_model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = dino_model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    scheduler.step()

    # Validation
    dino_model.eval()
    v_preds, v_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            preds = dino_model(imgs).argmax(1).cpu()
            v_preds.extend(preds.tolist()); v_labels.extend(labels.tolist())
    val_acc = sum(p == l for p, l in zip(v_preds, v_labels)) / len(v_labels)
    val_f1 = f1_score(v_labels, v_preds, average='macro', zero_division=0)

    if dino_early.step(val_f1, epoch):
        torch.save(dino_model.state_dict(), dino_best_path)
        status = '⭐ [BEST SAVED]'
    else:
        status = f'[Patience: {dino_early.counter}/{dino_early.patience}]'
    print(f'[DINOv2] Epoch {epoch:02d}/12 | Loss: {total_loss/total:.4f} | ValAcc: {val_acc:.3f} | ValMacroF1: {val_f1:.4f} {status}')
    if dino_early.early_stop:
        print(f'🛑 Early stopping triggered at epoch {epoch}. Best Val F1: {dino_early.best_score:.4f}')
        break


## SECTION 08 — Evaluation & Per-Class Macro F1
Evaluates the best checkpoint on the unseen Test set.

In [ ]:
# ============================================================
# SECTION 08 — Test Set Evaluation
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def evaluate_checkpoint(model, ckpt_path, name):
    if not os.path.exists(ckpt_path): return
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    t_preds, t_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            preds = model(imgs).argmax(1).cpu()
            t_preds.extend(preds.tolist()); t_labels.extend(labels.tolist())
    acc = sum(p == l for p, l in zip(t_preds, t_labels)) / len(t_labels)
    f1  = f1_score(t_labels, t_preds, average='macro', zero_division=0)
    print(f'\n=== {name} Test Set Performance ===')
    print(f'Accuracy: {acc:.4f} | Macro F1: {f1:.4f}')
    print(classification_report(t_labels, t_preds, target_names=TARGET_CLASSES, zero_division=0))
    cm = confusion_matrix(t_labels, t_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES, cmap='Greens')
    plt.title(f'{name} — Test Confusion Matrix')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

evaluate_checkpoint(mobilenet, mb_best_path, 'MobileNetV3 (Fast)')
evaluate_checkpoint(dino_model, dino_best_path, 'DINOv2 ViT-S/14 (High Accuracy)')


## SECTION 09 — ONNX Export & Class Map Synchronization
Exports models to ONNX and saves `class_index.json` to guarantee 100% label alignment.

In [ ]:
# ============================================================
# SECTION 09 — Export ONNX & Save Class Map
# ============================================================
class_map_path = os.path.join(MODEL_DIR, 'class_index.json')
with open(class_map_path, 'w', encoding='utf-8') as f:
    json.dump(IDX_TO_CLASS, f, indent=2)
print(f'✅ class_index.json synchronized at: {class_map_path}')

dummy_input = torch.randn(1, 3, 224, 224).to(device)
onnx_mb_path = os.path.join(MODEL_DIR, 'mobilenetv3.onnx')
torch.onnx.export(
    mobilenet, dummy_input, onnx_mb_path,
    input_names=['image'], output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=14
)
print(f'✅ Exported MobileNetV3 ONNX: {onnx_mb_path}')
